# StageBridge: Receiver-Centered Niche Encoding for Cell State Transitions

This notebook demonstrates model inference, attention extraction, and analysis of learned niche representations. The central hypothesis is that cross-sectional cell state transitions become more identifiable when a focal receiver cell is modeled together with its local microenvironmental niche.

**Contents:**
1. Load semi-synthetic benchmark with ground-truth interactions
2. Load trained model and run inference
3. Extract and visualize attention weights
4. Analyze attention alignment with ground-truth sender locations
5. Weight analysis and ablation results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import json
from pathlib import Path
from scipy.spatial.distance import cdist

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
})

RESULTS_DIR = Path('results/v1')
DATA_DIR = Path('data')
FIG_DIR = Path('figures/publication')
FIG_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Semi-Synthetic Benchmark

The semi-synthetic benchmark provides ground-truth interaction labels for rigorous evaluation. Following the AMICI methodology (Hong et al. 2025), spatial proximity between sender and receiver cells is explicitly controlled. Nine interaction rules are defined based on established ligand-receptor pairs from lung cancer biology, including IL1B-IL1R1 (macrophage to epithelial, 50um range) and CXCL12-CXCR4 (fibroblast to epithelial, 100um range).

In [ ]:
# Load semi-synthetic benchmark
benchmark_size = 'medium'  # 'small' (1K), 'medium' (5K), 'large' (20K)
benchmark_dir = DATA_DIR / f'semisynthetic_benchmark_{benchmark_size}'

coords = pd.read_parquet(benchmark_dir / 'coordinates.parquet')
neighborhoods = pd.read_parquet(benchmark_dir / 'neighborhoods.parquet')
labels = pd.read_parquet(benchmark_dir / 'ground_truth_labels_fixed.parquet')

with open(benchmark_dir / 'summary.json') as f:
    summary = json.load(f)

print(f'Benchmark: {benchmark_size}')
print(f'Total cells: {len(coords)}')
print(f'Interacting: {labels["is_interacting"].sum()} ({100*labels["is_interacting"].mean():.1f}%)')
print(f'\nExpected attention ring distribution:')
print(labels[labels['is_interacting']]['expected_attention_ring'].value_counts().sort_index())

In [ ]:
# Visualize spatial layout and ground-truth
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel A: Cell types
ax = axes[0]
cell_type_colors = {
    'AT2': '#1f77b4', 'Basal': '#ff7f0e', 'Ciliated': '#2ca02c', 'Secretory': '#d62728',
    'Macrophages': '#9467bd', 'Fibroblast lineage': '#8c564b', 'T cell lineage': '#e377c2',
    'Capillary': '#7f7f7f', 'Mast cells': '#bcbd22'
}
for ct in coords['cell_type'].unique():
    mask = coords['cell_type'] == ct
    ax.scatter(coords.loc[mask, 'x'], coords.loc[mask, 'y'],
               c=cell_type_colors.get(ct, 'gray'), s=8, alpha=0.6, label=ct)
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
ax.set_title('A. Cell Type Distribution')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)

# Panel B: Interacting cells
ax = axes[1]
non_int = ~labels['is_interacting'].values
ax.scatter(coords.loc[non_int, 'x'], coords.loc[non_int, 'y'],
           c='#e0e0e0', s=6, alpha=0.4, label='Non-interacting')
ax.scatter(coords.loc[labels['is_interacting'].values, 'x'],
           coords.loc[labels['is_interacting'].values, 'y'],
           c='#c0392b', s=12, alpha=0.7, label='Interacting')
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
ax.set_title('B. Ground-Truth Interactions')
ax.legend()

# Panel C: Expected ring distribution
ax = axes[2]
ring_counts = labels[labels['is_interacting']]['expected_attention_ring'].value_counts().sort_index()
ring_labels = ['Ring 1\n(0-50um)', 'Ring 2\n(50-100um)', 'Ring 3\n(100-150um)', 'Ring 4\n(150-200um)']
colors = ['#c0392b', '#e74c3c', '#f39c12', '#f1c40f']
bars = ax.bar(range(len(ring_counts)), ring_counts.values, color=colors[:len(ring_counts)])
ax.set_xticks(range(len(ring_counts)))
ax.set_xticklabels([ring_labels[i] for i in ring_counts.index])
ax.set_ylabel('Count')
ax.set_title('C. Expected Attention Ring')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_semisynthetic_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Load Trained Model

The trained checkpoint contains the full StageBridge architecture with learned weights for niche encoding and transition prediction.

In [ ]:
# Load checkpoint
checkpoint_path = RESULTS_DIR / 'full/fold_0_seed_44_best_checkpoint.pt'
if not checkpoint_path.exists():
    # Fallback to other checkpoints
    alt_paths = list(RESULTS_DIR.glob('full/fold_*/seed_*/weights/final_model.pt'))
    if alt_paths:
        checkpoint_path = alt_paths[0]
    else:
        raise FileNotFoundError(f'No checkpoint found in {RESULTS_DIR}')

print(f'Loading: {checkpoint_path}')
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
state = checkpoint['model_state_dict']

print(f'\nCheckpoint keys: {list(checkpoint.keys())}')
if 'config' in checkpoint:
    print('\nModel config:')
    config = checkpoint['config']
    if isinstance(config, dict):
        model_config = config.get('model_config', config)
        for k, v in model_config.items():
            print(f'  {k}: {v}')

In [ ]:
# Build model from checkpoint
from stagebridge.models.stagebridge import StageBridge, StageBridgeConfig

# Infer config from state dict
config = StageBridgeConfig.from_checkpoint(checkpoint)
model = StageBridge(config)
model.load_state_dict(state)
model = model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {n_params:,} parameters')
print(f'Config: hidden_dim={config.hidden_dim}, num_heads={config.num_heads}')

## 3. Prepare Data and Run Inference

Convert the neighborhoods dataframe to tensor batches and run inference to extract embeddings and attention weights.

In [ ]:
from stagebridge.contracts import LATENT_DIM, HLCA_DIM, LUCA_DIM, MAX_CELLS_PER_RING

def prepare_batch(neighborhoods_df, labels_df=None, max_cells=MAX_CELLS_PER_RING, device='cpu'):
    """Convert neighborhoods dataframe to model input tensors."""
    n = len(neighborhoods_df)
    
    # Receiver embeddings
    receivers = np.stack(neighborhoods_df['receiver_z'].values).astype(np.float32)
    hlca_z = np.stack(neighborhoods_df['hlca_z'].values).astype(np.float32)
    luca_z = np.stack(neighborhoods_df['luca_z'].values).astype(np.float32)
    
    # Ring cells and masks
    ring_cells = []
    ring_masks = []
    
    for ring_idx in range(4):
        col = f'ring_{ring_idx+1}_cells'
        padded = []
        masks = []
        
        for cells in neighborhoods_df[col].tolist():
            cells = np.array(cells) if len(cells) > 0 else np.zeros((0, LATENT_DIM))
            n_cells = len(cells)
            
            pad = np.zeros((max_cells, LATENT_DIM), dtype=np.float32)
            mask = np.zeros(max_cells, dtype=bool)
            
            if n_cells > 0:
                n_use = min(n_cells, max_cells)
                pad[:n_use] = cells[:n_use, :LATENT_DIM]
                mask[:n_use] = True
            
            padded.append(pad)
            masks.append(mask)
        
        ring_cells.append(torch.tensor(np.array(padded)).to(device))
        ring_masks.append(torch.tensor(np.array(masks)).to(device))
    
    # Stage indices
    stage_map = {'Normal': 0, 'AAH': 1, 'AIS': 2, 'MIA': 3, 'LUAD': 4}
    stage_idx = [stage_map.get(s, 0) for s in neighborhoods_df['stage']]
    
    batch = {
        'receiver': torch.tensor(receivers).to(device),
        'hlca': torch.tensor(hlca_z).to(device),
        'luca': torch.tensor(luca_z).to(device),
        'ring_cells': ring_cells,
        'ring_masks': ring_masks,
        'stage_idx': torch.tensor(stage_idx, dtype=torch.long).to(device),
    }
    
    if labels_df is not None:
        batch['is_interacting'] = labels_df['is_interacting'].values
        batch['expected_ring'] = labels_df['expected_attention_ring'].values
    
    return batch

# Prepare full dataset
batch = prepare_batch(neighborhoods, labels, device=device)
print(f'Receiver shape: {batch["receiver"].shape}')
print(f'Ring 1 cells shape: {batch["ring_cells"][0].shape}')

In [ ]:
# Register hooks to capture attention weights
attention_weights = {}

def get_attention_hook(name):
    def hook(module, input, output):
        if isinstance(output, tuple) and len(output) >= 2:
            attn = output[1]
            if attn is not None:
                attention_weights[name] = attn.detach().cpu()
    return hook

hooks = []
for name, module in model.named_modules():
    if isinstance(module, nn.MultiheadAttention):
        print(f'Hooking: {name}')
        hooks.append(module.register_forward_hook(get_attention_hook(name)))

In [ ]:
# Run inference in batches
BATCH_SIZE = 256
n_samples = len(neighborhoods)

all_embeddings = []
all_context = []
all_ring_attention = []

with torch.no_grad():
    for start in range(0, n_samples, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_samples)
        attention_weights.clear()
        
        # Slice batch
        b_receiver = batch['receiver'][start:end]
        b_hlca = batch['hlca'][start:end]
        b_luca = batch['luca'][start:end]
        b_ring_cells = [rc[start:end] for rc in batch['ring_cells']]
        b_ring_masks = [rm[start:end] for rm in batch['ring_masks']]
        b_stage = batch['stage_idx'][start:end]
        
        # Encode niche context
        niche_output = model.encode_niche(
            receiver=b_receiver,
            ring_cells=b_ring_cells,
            ring_masks=b_ring_masks,
            hlca=b_hlca,
            luca=b_luca,
        )
        
        all_embeddings.append(b_receiver.cpu())
        all_context.append(niche_output.context.cpu())
        
        # Extract ring attention from context_refiner
        for name, attn in attention_weights.items():
            if 'context_refiner' in name:
                # Attention shape: [batch, n_tokens, n_tokens]
                # Token order: [Receiver, Ring1-4, HLCA, LuCA, Pathway, Stats]
                # Extract receiver (query) attention to ring tokens (keys)
                ring_attn = attn[:, 0, 1:5]  # Receiver attending to rings 1-4
                all_ring_attention.append(ring_attn)
        
        if start == 0:
            print(f'Batch 0 attention keys: {list(attention_weights.keys())}')

print(f'\nProcessed {n_samples} cells')
print(f'Captured {len(all_ring_attention)} attention batches')

In [ ]:
# Clean up hooks
for h in hooks:
    h.remove()

# Concatenate results
embeddings = torch.cat(all_embeddings, dim=0).numpy()
context = torch.cat(all_context, dim=0).numpy()

if all_ring_attention:
    ring_attention = torch.cat(all_ring_attention, dim=0).numpy()
    print(f'Ring attention shape: {ring_attention.shape}')
else:
    print('No ring attention captured - model may not expose attention weights')
    # Create dummy attention for visualization
    ring_attention = np.ones((n_samples, 4)) / 4

## 4. Analyze Attention vs Ground Truth

Compare model attention patterns to ground-truth sender locations. If the model correctly identifies interaction partners, attention should concentrate on rings containing senders.

In [ ]:
# Attention alignment analysis
is_interacting = batch['is_interacting']
expected_ring = batch['expected_ring']

# For interacting cells, check if max attention is on expected ring
interacting_mask = is_interacting
interacting_attention = ring_attention[interacting_mask]
interacting_expected = expected_ring[interacting_mask]

# Predicted ring = argmax of attention
predicted_ring = np.argmax(interacting_attention, axis=1)
correct = (predicted_ring == interacting_expected)

print('Attention Alignment Analysis')
print('=' * 50)
print(f'Interacting cells: {interacting_mask.sum()}')
print(f'Correct ring prediction: {correct.sum()} ({100*correct.mean():.1f}%)')
print(f'Random baseline: {100/4:.1f}%')
print(f'\nPer-ring accuracy:')
for ring in range(4):
    ring_mask = interacting_expected == ring
    if ring_mask.sum() > 0:
        acc = (predicted_ring[ring_mask] == ring).mean()
        print(f'  Ring {ring+1}: {100*acc:.1f}% ({ring_mask.sum()} cells)')

In [ ]:
# Visualize attention patterns
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel A: Attention heatmap for interacting vs non-interacting
ax = axes[0, 0]
attn_by_group = {
    'Interacting': ring_attention[interacting_mask].mean(axis=0),
    'Non-interacting': ring_attention[~interacting_mask].mean(axis=0),
}
attn_df = pd.DataFrame(attn_by_group, index=['Ring 1', 'Ring 2', 'Ring 3', 'Ring 4']).T
sns.heatmap(attn_df, annot=True, fmt='.3f', cmap='Reds', ax=ax,
            cbar_kws={'label': 'Mean Attention'})
ax.set_title('A. Mean Attention by Cell Group')

# Panel B: Attention distribution by expected ring
ax = axes[0, 1]
ring_names = ['Ring 1', 'Ring 2', 'Ring 3', 'Ring 4']
for expected in range(4):
    mask = interacting_expected == expected
    if mask.sum() > 0:
        attn = interacting_attention[mask, expected]  # Attention to correct ring
        ax.boxplot([attn], positions=[expected], widths=0.6)
ax.set_xticks(range(4))
ax.set_xticklabels(ring_names)
ax.set_ylabel('Attention Weight')
ax.set_title('B. Attention to Correct Ring\n(interacting cells only)')
ax.axhline(0.25, ls='--', color='gray', alpha=0.5, label='Uniform')
ax.legend()

# Panel C: Confusion matrix
ax = axes[1, 0]
conf_matrix = np.zeros((4, 4))
for expected in range(4):
    mask = interacting_expected == expected
    if mask.sum() > 0:
        for predicted in range(4):
            conf_matrix[expected, predicted] = (predicted_ring[mask] == predicted).sum()
conf_matrix = conf_matrix / conf_matrix.sum(axis=1, keepdims=True)  # Normalize
sns.heatmap(conf_matrix, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=ring_names, yticklabels=ring_names)
ax.set_xlabel('Predicted Ring (argmax attention)')
ax.set_ylabel('Expected Ring (ground truth)')
ax.set_title('C. Ring Prediction Confusion Matrix')

# Panel D: Attention vs distance
ax = axes[1, 1]
ring_distances = [25, 75, 125, 175]  # Ring center distances in um
mean_attn = ring_attention.mean(axis=0)
std_attn = ring_attention.std(axis=0)
ax.errorbar(ring_distances, mean_attn, yerr=std_attn, marker='o', capsize=5, color='#c0392b')
ax.set_xlabel('Distance from Receiver (um)')
ax.set_ylabel('Mean Attention Weight')
ax.set_title('D. Attention Decay with Distance')
ax.set_xticks(ring_distances)
ax.axhline(0.25, ls='--', color='gray', alpha=0.5)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_attention_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Weight Analysis

The drift head implements a gated combination of two pathways: a latent-only path predicting from receiver state alone, and a context path using cross-attention over niche tokens. Comparing weight magnitudes reveals which pathway dominates.

In [ ]:
# Drift head pathway analysis
latent_only_norm = state['drift_head.latent_only.0.weight'].norm().item()
context_out_norm = state['drift_head.context_out_proj.weight'].norm().item()
gate_bias = state['drift_head.context_gate.2.bias'].item()

print('Drift Head Pathway Analysis')
print('=' * 50)
print(f'Latent-only path weight norm: {latent_only_norm:.4f}')
print(f'Context path weight norm:     {context_out_norm:.4f}')
print(f'Ratio (latent/context):       {latent_only_norm/context_out_norm:.2f}x')
print(f'\nContext gate bias: {gate_bias:.4f}')
print(f'Default gate: sigmoid({gate_bias:.3f}) = {torch.sigmoid(torch.tensor(gate_bias)).item():.3f}')

In [ ]:
# Ring differentiation analysis
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel A: Token projection weights
ax = axes[0, 0]
token_norms = {
    'Receiver': state['niche_tokenizer.token_proj.weight'].norm().item(),
    'HLCA': state['niche_tokenizer.hlca_proj.weight'].norm().item(),
    'LuCA': state['niche_tokenizer.luca_proj.weight'].norm().item(),
}
if 'niche_tokenizer.stats_proj.weight' in state:
    token_norms['Stats'] = state['niche_tokenizer.stats_proj.weight'].norm().item()
for i in range(4):
    token_norms[f'Ring {i+1}'] = state[f'niche_tokenizer.ring_poolers.{i}.proj.weight'].norm().item()

colors = ['#27ae60', '#2980b9', '#8e44ad', '#7f8c8d', '#c0392b', '#e74c3c', '#f39c12', '#f1c40f']
ax.bar(token_norms.keys(), token_norms.values(), color=colors[:len(token_norms)])
ax.set_ylabel('Projection Weight Norm')
ax.set_title('A. Token Input Projection Strength')
ax.tick_params(axis='x', rotation=45)

# Panel B: Ring PMA seed similarity
ax = axes[0, 1]
seeds = torch.stack([state[f'niche_tokenizer.ring_poolers.{i}.pma.seed_vectors'].squeeze() for i in range(4)])
seeds_norm = seeds / seeds.norm(dim=1, keepdim=True)
sim = (seeds_norm @ seeds_norm.T).numpy()
sns.heatmap(sim, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=['R1', 'R2', 'R3', 'R4'],
            yticklabels=['R1', 'R2', 'R3', 'R4'], ax=ax)
ax.set_title('B. Ring PMA Seed Similarity\n(low = differentiated)')

# Panel C: Drift head pathway comparison
ax = axes[1, 0]
bars = ax.bar(['Latent-only\n(no context)', 'Context\n(cross-attention)'],
              [latent_only_norm, context_out_norm], color=['#27ae60', '#c0392b'])
ax.set_ylabel('Weight Norm')
ax.set_title('C. Drift Head Pathway Weights')
for bar, val in zip(bars, [latent_only_norm, context_out_norm]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}', ha='center', fontsize=10)

# Panel D: Gate input decomposition
ax = axes[1, 1]
gate_input = state['drift_head.context_gate.0.weight']
# Input: [z_receiver, z_context, stage_embed]
hidden_dim = config.hidden_dim
gate_parts = {
    'Receiver': gate_input[:, :hidden_dim].norm().item(),
    'Context': gate_input[:, hidden_dim:2*hidden_dim].norm().item(),
    'Stage': gate_input[:, 2*hidden_dim:].norm().item(),
}
ax.bar(gate_parts.keys(), gate_parts.values(), color=['#27ae60', '#c0392b', '#3498db'])
ax.set_ylabel('Weight Norm')
ax.set_title('D. Gate Input Weights')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_model_weights.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Ablation Study Results

In [ ]:
# Load ablation results
report_path = RESULTS_DIR / 'comparison_report.json'
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    
    full_loss = report['full_model']['mean_val_loss']
    full_std = report['full_model']['std_val_loss']
    
    print('Full Model Performance')
    print('=' * 50)
    print(f'Validation Loss: {full_loss:.6f} +/- {full_std:.6f}')
    
    print('\nAblation Results (sorted by impact)')
    print('=' * 50)
    ablations = sorted(report['ablations'].items(), key=lambda x: x[1]['delta_vs_full'], reverse=True)
    for name, data in ablations:
        delta = data['delta_vs_full']
        print(f'{name:25s}: {data["mean_val_loss"]:.6f} ({delta:+.1f}%)')
else:
    print('No ablation report found')
    report = None

In [ ]:
if report:
    # Ablation visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel A: Violin plot
    ax = axes[0]
    rows = [{'config': 'Full Model', 'loss': loss} for loss in report['full_model']['all_losses']]
    for name, data in ablations:
        label = name.replace('_', ' ').title()
        rows.extend([{'config': label, 'loss': loss} for loss in data['all_losses']])
    df = pd.DataFrame(rows)
    
    order = ['Full Model'] + [name.replace('_', ' ').title() for name, _ in ablations]
    palette = {'Full Model': '#27ae60'}
    for name, data in ablations:
        label = name.replace('_', ' ').title()
        delta = data['delta_vs_full']
        palette[label] = '#c0392b' if delta > 8 else '#e74c3c' if delta > 4 else '#f39c12' if delta > 0 else '#3498db'
    
    sns.violinplot(data=df, x='config', y='loss', order=order, palette=palette, ax=ax, inner='box', cut=0)
    ax.axhline(full_loss, color='#27ae60', ls='--', lw=1.5, alpha=0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Validation Loss (MSE)')
    ax.set_title('A. Ablation Study')
    ax.tick_params(axis='x', rotation=45)
    
    # Panel B: Delta bar chart
    ax = axes[1]
    names = [name.replace('_', ' ').title() for name, _ in ablations]
    deltas = [data['delta_vs_full'] for _, data in ablations]
    colors = [palette[n] for n in names]
    
    bars = ax.barh(range(len(names)), deltas, color=colors)
    ax.axvline(0, color='black', lw=1)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    ax.set_xlabel('% Change vs Full Model')
    ax.set_title('B. Component Impact')
    ax.invert_yaxis()
    
    for bar, delta in zip(bars, deltas):
        x_pos = delta + 0.3 if delta >= 0 else delta - 0.3
        ha = 'left' if delta >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{delta:+.1f}%',
                va='center', ha=ha, fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig_ablation_study.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Summary and Conclusions

This analysis reveals several key findings about receiver-centered niche encoding for transition modeling.

In [ ]:
print('Summary of Findings')
print('=' * 60)
print()
print('1. ATTENTION ALIGNMENT')
print(f'   Ring prediction accuracy: {100*correct.mean():.1f}% (random: 25%)')
print(f'   Interacting cells tested: {interacting_mask.sum()}')
print()
print('2. PATHWAY ANALYSIS')
print(f'   Latent-only / Context ratio: {latent_only_norm/context_out_norm:.1f}x')
print(f'   Default gate value: {torch.sigmoid(torch.tensor(gate_bias)).item():.2f}')
print()
print('3. RING DIFFERENTIATION')
off_diag = sim[np.triu_indices(4, k=1)]
print(f'   Mean off-diagonal similarity: {off_diag.mean():.3f}')
print(f'   Rings are {"differentiated" if off_diag.mean() < 0.5 else "similar"}')
print()
if report:
    print('4. ABLATION RANKING')
    for i, (name, data) in enumerate(ablations[:5], 1):
        print(f'   {i}. {name.replace("_", " ").title()}: {data["delta_vs_full"]:+.1f}%')
print()
print('5. INTERPRETATION')
print('   - The gating mechanism is the most critical component')
print('   - Model CAN differentiate rings but learned to down-weight context')
print('   - Data signal may be predominantly cell-intrinsic')

In [ ]:
# Generate combined summary figure
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# Row 1: Semi-synthetic data
ax = fig.add_subplot(gs[0, 0])
non_int = ~labels['is_interacting'].values
ax.scatter(coords.loc[non_int, 'x'], coords.loc[non_int, 'y'], c='#e0e0e0', s=4, alpha=0.3)
ax.scatter(coords.loc[labels['is_interacting'].values, 'x'],
           coords.loc[labels['is_interacting'].values, 'y'], c='#c0392b', s=8, alpha=0.6)
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
ax.set_title(f'A. Ground-Truth Interactions\n({interacting_mask.sum()} cells)')

# Attention heatmap
ax = fig.add_subplot(gs[0, 1])
sns.heatmap(attn_df, annot=True, fmt='.3f', cmap='Reds', ax=ax)
ax.set_title('B. Mean Ring Attention')

# Confusion matrix
ax = fig.add_subplot(gs[0, 2])
sns.heatmap(conf_matrix, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=['R1', 'R2', 'R3', 'R4'], yticklabels=['R1', 'R2', 'R3', 'R4'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Expected')
ax.set_title('C. Ring Prediction Matrix')

# Row 2: Model weights
ax = fig.add_subplot(gs[1, 0])
ax.bar(['Latent-only', 'Context'], [latent_only_norm, context_out_norm], color=['#27ae60', '#c0392b'])
ax.set_ylabel('Weight Norm')
ax.set_title('D. Drift Head Pathways')

ax = fig.add_subplot(gs[1, 1])
sns.heatmap(sim, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=['R1', 'R2', 'R3', 'R4'], yticklabels=['R1', 'R2', 'R3', 'R4'], ax=ax)
ax.set_title('E. Ring Differentiation')

ax = fig.add_subplot(gs[1, 2])
ax.errorbar(ring_distances, mean_attn, yerr=std_attn, marker='o', capsize=5, color='#c0392b')
ax.axhline(0.25, ls='--', color='gray', alpha=0.5)
ax.set_xlabel('Distance (um)')
ax.set_ylabel('Attention')
ax.set_title('F. Attention vs Distance')

# Row 3: Ablations and summary
if report:
    ax = fig.add_subplot(gs[2, :2])
    sns.violinplot(data=df, x='config', y='loss', order=order, palette=palette, ax=ax, inner='box', cut=0)
    ax.axhline(full_loss, color='#27ae60', ls='--', lw=1.5, alpha=0.7)
    ax.set_xlabel('')
    ax.set_ylabel('Validation Loss')
    ax.set_title('G. Ablation Study')
    ax.tick_params(axis='x', rotation=45)

# Summary table
ax = fig.add_subplot(gs[2, 2])
ax.axis('off')
table_data = [
    ['Metric', 'Value'],
    ['Ring accuracy', f'{100*correct.mean():.1f}%'],
    ['Pathway ratio', f'{latent_only_norm/context_out_norm:.1f}x'],
    ['Ring similarity', f'{off_diag.mean():.2f}'],
]
if report:
    table_data.extend([
        ['No-gate impact', f'+{ablations[0][1]["delta_vs_full"]:.1f}%'],
        ['No-niche impact', f'+{report["ablations"].get("no_niche", {}).get("delta_vs_full", 0):.1f}%'],
    ])
table = ax.table(cellText=table_data, loc='center', cellLoc='left', colWidths=[0.5, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)
ax.set_title('H. Summary')

plt.savefig(FIG_DIR / 'fig_combined_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFigures saved to {FIG_DIR}/')